# 🚀 Mistral Query Router — Fine-tuning on Colab
Fine-tune **Ministral 3B** to classify queries into model tiers (1-4).

**Before running:** Go to Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Step 1: Install dependencies
!pip install -q torch transformers datasets accelerate peft bitsandbytes trl wandb huggingface_hub

In [ ]:
# Step 2: Set your API keys (fill these in!)
import os
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'\nos.environ['WANDB_API_KEY'] = 'YOUR_WANDB_API_KEY_HERE'\n
# Login to HF
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

In [ ]:
# Step 3: Verify GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Step 4: Config
BASE_MODEL = 'mistralai/Ministral-3-3B-Instruct-2512'
DATASET_ID = 'mistral-hackaton-2026/router-training-data'
OUTPUT_REPO = 'mistral-hackaton-2026/mistral-query-router'
WANDB_PROJECT = 'mistral-router'

# Hyperparameters
LEARNING_RATE = 2e-4
EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
MAX_SEQ_LENGTH = 512
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

In [ ]:
# Step 5: Init W&B
import wandb
from datetime import datetime

wandb.login(key=os.environ['WANDB_API_KEY'])
wandb.init(
    project=WANDB_PROJECT,
    name=f'router-{datetime.now().strftime("%Y%m%d_%H%M%S")}',
    config={
        'base_model': BASE_MODEL,
        'lora_rank': LORA_RANK,
        'learning_rate': LEARNING_RATE,
        'epochs': EPOCHS,
    },
)

In [ ]:
# Step 6: Load tokenizer + model with QLoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=os.environ['HF_TOKEN'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model (this takes ~2 min)...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=os.environ['HF_TOKEN'],
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print(f'Model loaded! Parameters: {model.num_parameters():,}')

In [ ]:
# Step 7: Load and format dataset
from datasets import load_dataset

print('Loading dataset...')
dataset = load_dataset(DATASET_ID, token=os.environ['HF_TOKEN'])

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

dataset = dataset.map(format_chat)
print(f'Train: {len(dataset["train"])} examples')
if 'validation' in dataset:
    print(f'Val:   {len(dataset["validation"])} examples')
elif 'val' in dataset:
    print(f'Val:   {len(dataset["val"])} examples')

# Preview one example
print('\n--- Sample ---')
print(dataset['train'][0]['text'][:500])

In [ ]:
# Step 8: Configure LoRA + Training
from trl import SFTTrainer, SFTConfig

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

val_dataset = dataset.get('validation', dataset.get('val'))

training_args = SFTConfig(
    output_dir='./output',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=5,
    eval_strategy='epoch' if val_dataset else 'no',
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    report_to='wandb',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=val_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print(f'Ready to train! Estimated steps: {len(dataset["train"]) * EPOCHS // (BATCH_SIZE * GRADIENT_ACCUMULATION)}')

In [ ]:
# Step 9: TRAIN! 🚀
print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# Step 10: Save and push to HuggingFace Hub
trainer.save_model('./output')
tokenizer.save_pretrained('./output')

# Push to your org repo
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(OUTPUT_REPO, exist_ok=True)
api.upload_folder(
    folder_path='./output',
    repo_id=OUTPUT_REPO,
    token=os.environ['HF_TOKEN'],
)
print(f'\n✅ Model pushed to: https://huggingface.co/{OUTPUT_REPO}')

In [ ]:
# Step 11: Quick test
test_queries = [
    'What is 2+2?',
    'Explain the difference between weather and climate.',
    'Debug this async Node.js code for memory leaks.',
    'Design a distributed database for a social media platform.',
]

from peft import PeftModel

SYSTEM_PROMPT = '''You are a query complexity router for Mistral AI models. Given a user query, classify it into the appropriate model tier and provide a confidence score.

Output ONLY valid JSON in this exact format:
{"model_tier": <1|2|3|4>, "confidence": <0.0-1.0>}

Tier definitions:
- Tier 1 (small): Simple factual lookups, greetings, trivial math, yes/no questions
- Tier 2 (medium): Summarization, translation, moderate reasoning, simple coding
- Tier 3 (large): Complex analysis, multi-step reasoning, debugging, comparisons
- Tier 4 (xlarge): Expert-level research, novel problem-solving, proofs, system design'''

for q in test_queries:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': q},
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to('cuda')
    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens=50, temperature=0.1, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'Q: {q}')
    print(f'A: {response}\n')

In [ ]:
# Finish W&B
wandb.finish()
print('All done! Check your model at:')
print(f'  https://huggingface.co/{OUTPUT_REPO}')
print(f'  https://wandb.ai — project: {WANDB_PROJECT}')